# ASL Transfer Learning Pipeline

## Goal

Train and evaluate an ASL gesture classifier for three classes (`A`, `B`, `C`) using transfer learning with InceptionV3.

## Scope

This notebook presents one coherent pipeline: context, implementation, training procedure, evaluation, and rationale. It intentionally focuses on the current pipeline and on the reasoning behind its design choices.


## Problem Statement

The transfer-learning setup targets ASL recognition for three static gestures. A frozen ImageNet-pretrained backbone is used to extract visual features, and a lightweight classifier head maps those features to gesture classes.

## Current State Summary

| Aspect | Current State |
| --- | --- |
| Model | InceptionV3 backbone + GAP + Dense(256) + Dropout(0.3) + Softmax |
| Classes | `A`, `B`, `C` |
| Input size | `299x299` |
| Data split | `Data/split/train` (development pool) and `Data/split/test` with validation derived from train |
| Training strategy | Two-stage transfer learning |
| Stage 1 | Frozen backbone, classifier head only |
| Stage 2 | Fine-tune top 30 backbone layers |
| Checkpoint format | `.keras` |

## Review Notes

- Class count is currently fixed to three classes because the dataset folders expose `A`, `B`, and `C`.
- A dedicated test split is required because accuracy on tiny ad hoc samples is not meaningful.
- Fine-tuning is much faster with CUDA enabled, but the pipeline still works on CPU.


In [4]:
import os
import sys
from pathlib import Path


cwd = Path.cwd().resolve()
for candidate in [cwd, *cwd.parents]:
    src_dir = os.path.join(str(candidate), "src")
    if os.path.exists(os.path.join(src_dir, "utils")):
        BASE_DIR = src_dir
        break
else:
    raise RuntimeError("Could not find project src directory")

if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

DATA_DIR = os.path.join(BASE_DIR, "Data", "collected")
print(f"Input data directory: {DATA_DIR}")

def running_in_notebook() -> bool:
    try:
        from IPython import get_ipython
        shell = get_ipython()
        return shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell"
    except Exception:
        return False


def ensure_cuda_library_path():
    if os.environ.get("TF_CUDA_LIBS_READY") == "1":
        return

    site_packages = os.path.join(
        sys.prefix,
        "lib",
        f"python{sys.version_info.major}.{sys.version_info.minor}",
        "site-packages",
        "nvidia",
    )
    lib_dirs = [
        os.path.join(site_packages, "cuda_runtime", "lib"),
        os.path.join(site_packages, "cudnn", "lib"),
        os.path.join(site_packages, "cublas", "lib"),
        os.path.join(site_packages, "cufft", "lib"),
        os.path.join(site_packages, "curand", "lib"),
        os.path.join(site_packages, "cusolver", "lib"),
        os.path.join(site_packages, "cusparse", "lib"),
        os.path.join(site_packages, "nccl", "lib"),
        os.path.join(site_packages, "nvjitlink", "lib"),
    ]
    existing_dirs = [path for path in lib_dirs if os.path.exists(path)]

    if not existing_dirs:
        return

    current_ld_path = os.environ.get("LD_LIBRARY_PATH", "")
    current_entries = [entry for entry in current_ld_path.split(":") if entry]
    new_entries = [entry for entry in existing_dirs if entry not in current_entries]
    if not new_entries:
        return

    os.environ["LD_LIBRARY_PATH"] = ":".join(new_entries + current_entries)
    os.environ["TF_CUDA_LIBS_READY"] = "1"

    if running_in_notebook():
        print(
            "Updated LD_LIBRARY_PATH for CUDA libs. Restart Jupyter kernel once before importing TensorFlow if GPU is not detected."
        )
        return

    os.execvpe(sys.executable, [sys.executable] + sys.argv, os.environ)


ensure_cuda_library_path()



Input data directory: /home/sleepyy/Desktop/aulas/3ano2sem/IAA/proj_IAA/src/Data/collected


## Environment Decisions

- The notebook resolves `BASE_DIR` the same way as the handmarks notebook: walk upward until `src/utils` exists, then use that `src` directory as project base.
- This is simpler than maintaining separate repo-root and project-root rules, and it matches the actual project layout used here.
- `src` is added to `sys.path` so notebook execution can import project utilities without requiring a package install step.
- CUDA library discovery is kept in the notebook because TensorFlow GPU setups commonly fail when NVIDIA wheels exist but their library directories are missing from `LD_LIBRARY_PATH`.
- In notebook mode the code avoids `os.execvpe(...)` because replacing the current process kills the Jupyter kernel.
- In script mode a process restart is still valid because TensorFlow must see the corrected library path before it initializes.


In [5]:
IMG_SIZE = (299, 299)
BATCH_SIZE = 16
STAGE1_EPOCHS = 30
STAGE2_EPOCHS = 50
STAGE1_LR = 0.001
STAGE2_LR = 0.0001
FINE_TUNE_LAYERS = 30
SEED = 123

DATA_DIR_DEV = os.path.join(BASE_DIR, "Data", "split", "train")
DATA_DIR_TEST = os.path.join(BASE_DIR, "Data", "split", "test")
MODEL_SAVE_PATH = os.path.join(BASE_DIR, "models", "base_datasetABC_model.keras")

os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

print(f"Base dir:  {BASE_DIR}")
print(f"Dev dir:   {DATA_DIR_DEV}")
print(f"Test dir:  {DATA_DIR_TEST}")
print(f"Model:     {MODEL_SAVE_PATH}")


Base dir:  /home/sleepyy/Desktop/aulas/3ano2sem/IAA/proj_IAA/src
Train dir: /home/sleepyy/Desktop/aulas/3ano2sem/IAA/proj_IAA/src/Data/split/train
Dev dir:   /home/sleepyy/Desktop/aulas/3ano2sem/IAA/proj_IAA/src/Data/split/train (validation derived with validation_split=0.2)
Test dir:  /home/sleepyy/Desktop/aulas/3ano2sem/IAA/proj_IAA/src/Data/split/test
Model:     /home/sleepyy/Desktop/aulas/3ano2sem/IAA/proj_IAA/src/models/base_datasetABC_model.keras


## Hyperparameter And Path Decisions

### Input And Batching

- `IMG_SIZE = (299, 299)` because InceptionV3 is designed for that native input resolution.
- `BATCH_SIZE = 16` because it balances stable optimization with GPU memory limits and remains practical on weaker hardware.
- `SEED = 123` so dataset order and split-dependent behavior stay reproducible.

### Training Schedule

- `STAGE1_EPOCHS = 30` gives the classifier head enough room to converge without assuming a fixed early stopping point.
- `STAGE2_EPOCHS = 50` keeps a generous upper bound for fine-tuning while callbacks stop training once validation quality stops improving.
- `STAGE1_LR = 0.001` is appropriate when only the new head is learning.
- `STAGE2_LR = 0.0001` is 10x lower because partial unfreezing should adapt pretrained features gently, not overwrite them.
- `FINE_TUNE_LAYERS = 30` focuses adaptation on high-level features while preserving general low-level visual features.

### Dataset Layout

- Dedicated `train`, `val`, and `test` directories are used because a held-out test set is necessary for meaningful reporting.
- Data and model paths are resolved from `BASE_DIR` so all training assets consistently live under `src/Data` and `src/models`.
- The model is saved under `models/` in `.keras` format because that is the recommended modern Keras serialization format.


In [ ]:
import tensorflow as tf

def configure_device():
    gpus = tf.config.list_physical_devices("GPU")
    if not gpus:
        print("No GPU detected by TensorFlow. Training will run on CPU.")
        return

    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

    print(f"Using GPU: {[gpu.name for gpu in gpus]}")


configure_device()


NameError: name 'tf' is not defined

## Device Decision

- Memory growth is enabled on each GPU to avoid TensorFlow eagerly reserving all VRAM, which is a common source of notebook instability.
- CPU fallback is explicit so the notebook still runs when CUDA is unavailable.


## Process And Load Images

This section validates the image folders, loads sample files from disk, resizes them to the model input size, converts them into tensors, and previews representative examples. The goal is to make data issues visible before augmentation and training begin.


In [ ]:
import math
import random
import matplotlib.pyplot as plt


def list_class_image_files(data_dir):
    if not os.path.exists(data_dir):
        raise FileNotFoundError(f"Data directory not found: {data_dir}")

    class_map = {}
    for class_name in sorted(os.listdir(data_dir)):
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.isdir(class_dir):
            continue

        image_files = [
            os.path.join(class_dir, file_name)
            for file_name in sorted(os.listdir(class_dir))
            if file_name.lower().endswith((".png", ".jpg", ".jpeg"))
        ]
        class_map[class_name] = image_files

    return class_map


def load_and_process_image(image_path, target_size=IMG_SIZE):
    image = tf.keras.utils.load_img(image_path, target_size=target_size)
    image_array = tf.keras.utils.img_to_array(image)
    return image_array


dev_image_map = list_class_image_files(DATA_DIR_DEV)
test_image_map = list_class_image_files(DATA_DIR_TEST)

print("Image counts by split:")
for split_name, image_map in [("dev", dev_image_map), ("test", test_image_map)]:
    total_images = sum(len(files) for files in image_map.values())
    print(f"  {split_name}: {total_images} images")
    for class_name, files in image_map.items():
        print(f"    {class_name}: {len(files)}")


preview_samples = []
for class_name, files in train_image_map.items():
    if files:
        preview_samples.append((class_name, random.choice(files)))


if preview_samples:
    n_cols = min(3, len(preview_samples))
    n_rows = math.ceil(len(preview_samples) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    axes = axes.flat if hasattr(axes, "flat") else [axes]

    for ax, (class_name, image_path) in zip(axes, preview_samples):
        image_array = load_and_process_image(image_path)
        ax.imshow(image_array.astype("uint8"))
        ax.set_title(f"{class_name}\n{os.path.basename(image_path)}\nshape={image_array.shape}")
        ax.axis("off")

    for ax in list(axes)[len(preview_samples):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No training images found for preview.")


## Image Loading Decisions

- The image check appears before training because missing folders, empty classes, or unreadable files should fail fast.
- Sample images are resized to `299x299` during preview because that matches the downstream InceptionV3 input size.
- One random training image per class is enough to verify labels, crop quality, and file readability without cluttering the notebook.
- Split-level counts are printed because imbalance and missing classes are easier to detect from explicit numbers than from later metrics.


In [ ]:
def augment(image, label, img_size=IMG_SIZE):
    """Apply random augmentations to a single image.

    Horizontal flip is intentionally excluded because mirroring a hand sign
    can change its meaning or hand orientation.
    """
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.image.random_contrast(image, lower=0.8, upper=1.2)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)

    crop_factor = tf.random.uniform([], minval=0.8, maxval=1.0)
    orig_height = tf.shape(image)[0]
    orig_width = tf.shape(image)[1]
    crop_h = tf.cast(tf.cast(orig_height, tf.float32) * crop_factor, tf.int32)
    crop_w = tf.cast(tf.cast(orig_width, tf.float32) * crop_factor, tf.int32)

    image = tf.image.random_crop(image, size=[crop_h, crop_w, 3])
    image = tf.image.resize(image, img_size)

    pad_amount = int(0.1 * img_size[0])
    image = tf.image.resize_with_crop_or_pad(
        image,
        img_size[0] + pad_amount,
        img_size[1] + pad_amount,
    )
    image = tf.image.random_crop(image, size=[img_size[0], img_size[1], 3])
    image = tf.clip_by_value(image, 0.0, 255.0)

    return image, label


## Augmentation Decisions

- Augmentation is done on the fly so the effective training set grows without duplicating files on disk.
- Brightness and contrast variation simulate lighting changes from webcams and room conditions.
- Saturation variation improves robustness to camera auto-white-balance and color shifts.
- Zoom via crop-and-resize helps the model tolerate hands appearing at slightly different scales.
- Translation helps because hands will not always be perfectly centered.
- Horizontal flip is explicitly avoided because sign orientation is semantically important.
- Values are clipped back to `[0, 255]` so later normalization receives a valid image range.


In [ ]:
print("Loading datasets...")

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR_DEV,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR_DEV,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR_TEST,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_ds.class_names
n_classes = len(class_names)
print(f"Classes ({n_classes}): {class_names}")

train_ds = train_ds.unbatch()
train_ds = train_ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)


## Data Pipeline Decisions

- `image_dataset_from_directory(...)` is used because the dataset is already organized by class folders, which maps cleanly to TensorFlow's directory-based loader.
- Training data is shuffled because gradient-based training benefits from mixed sample order.
- Validation and test data are not shuffled because deterministic evaluation makes debugging and metric comparison easier.
- Training data is unbatched, augmented per image, then batched again so each example receives its own random transform.
- `num_parallel_calls=tf.data.AUTOTUNE` and `.prefetch(tf.data.AUTOTUNE)` are used to keep the input pipeline from starving the GPU or CPU trainer.
- Class names are read from the dataset itself so the output dimension matches the real folder structure instead of hard-coding assumptions.


In [ ]:
print("Building model...")

inception = tf.keras.applications.InceptionV3(
    weights="imagenet",
    input_shape=(299, 299, 3),
    include_top=False,
)
inception.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1),
    inception,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(n_classes, activation="softmax"),
])

model.summary()


## Architecture Decisions

### 1. Normalize Before InceptionV3

- The rescaling layer is placed before the backbone because InceptionV3 expects inputs in the `[-1, 1]` range.
- This choice keeps preprocessing inside the model graph, which makes saved inference behavior consistent with training behavior.

### 2. Use InceptionV3 With `include_top=False`

- The ImageNet classifier head is removed because ASL gesture classes do not match ImageNet labels.
- Keeping only the feature extractor preserves useful pretrained visual features while allowing a task-specific classifier head.

### 3. Freeze The Backbone Initially

- Stage 1 freezes InceptionV3 because the classifier head starts from random weights.
- This avoids large random gradients destroying pretrained features too early.

### 4. GlobalAveragePooling Instead Of Flatten

- `GlobalAveragePooling2D()` reduces the 2D feature maps to a compact representation with far fewer parameters than a large flatten layer.
- This is a strong regularization choice for a relatively small three-class problem.

### 5. Dense(256) + Dropout(0.3)

- `Dense(256)` gives the classifier head enough capacity to separate classes from a 2048-dimensional backbone feature vector without creating an oversized head.
- `Dropout(0.3)` adds regularization that is especially useful once augmentation and fine-tuning are introduced.

### 6. Softmax Output

- `Dense(n_classes, activation="softmax")` is the standard output layer for mutually exclusive multiclass classification.


In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=STAGE1_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)


## Optimization Decisions

- Adam is used because it is a reliable default optimizer for transfer learning with a small classifier head.
- `sparse_categorical_crossentropy` is the correct loss because labels are integer-encoded class IDs, not one-hot vectors.
- Accuracy is tracked because the task is balanced multiclass classification and accuracy is easy to interpret.
- `EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)` stops overtraining and preserves the best generalizing weights.
- `ReduceLROnPlateau(...)` reacts to stalled validation progress instead of assuming improvement timing in advance.
- `ModelCheckpoint(..., monitor="val_accuracy", save_best_only=True)` keeps the strongest validation checkpoint for later evaluation.
- Saving in `.keras` format is preferred because it is the modern Keras format and better preserves model state.


In [ ]:
print("\n" + "=" * 60)
print("STAGE 1: Training classifier head (Inception frozen)")
print("=" * 60)

history_stage1 = model.fit(
    train_ds,
    epochs=STAGE1_EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks,
)


## Stage 1 Decision

- The first stage trains only the classifier head so it can learn task-specific class boundaries without destabilizing the pretrained backbone.
- This is the standard transfer-learning recipe when the backbone starts from useful pretrained weights and the downstream dataset is much smaller than ImageNet.


In [ ]:
print("\n" + "=" * 60)
print(f"STAGE 2: Fine-tuning top {FINE_TUNE_LAYERS} InceptionV3 layers")
print("=" * 60)

for layer in inception.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False
for layer in inception.layers[-FINE_TUNE_LAYERS:]:
    layer.trainable = True

trainable_count = sum(1 for layer in inception.layers if layer.trainable)
total_count = len(inception.layers)
print(f"Trainable layers in InceptionV3: {trainable_count}/{total_count}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=STAGE2_LR),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

stage1_end_epoch = len(history_stage1.history["loss"])

history_stage2 = model.fit(
    train_ds,
    epochs=STAGE2_EPOCHS,
    initial_epoch=stage1_end_epoch,
    validation_data=val_ds,
    callbacks=callbacks,
)


## Fine-Tuning Decisions

### 1. Two-Stage Training

- Fine-tuning happens only after the head has learned a reasonable classifier, which reduces catastrophic forgetting.

### 2. Unfreeze Only Top 30 Layers

- Early layers capture generic low-level visual primitives such as edges and textures.
- Later layers are more task-specific, so adapting only the top portion is a better compute and overfitting tradeoff.

### 3. Recompile Before Stage 2

- Keras requires recompilation after changing trainable flags so the optimizer tracks the correct trainable variables.

### 4. Lower Learning Rate

- Fine-tuning uses `1e-4` because small updates are safer for pretrained weights than the stage-1 learning rate.

### 5. Continue Epoch Counting

- `initial_epoch=stage1_end_epoch` keeps stage numbering continuous in the training logs, which makes later analysis clearer.


In [ ]:
print("\n" + "=" * 60)
print("FINAL EVALUATION ON TEST SET")
print("=" * 60)

best_model = tf.keras.models.load_model(MODEL_SAVE_PATH)
test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=1)

print(f"\nTest Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Best model saved to {MODEL_SAVE_PATH}")


## Evaluation Decision

- The best checkpoint is reloaded before testing so the reported test score matches the strongest validation-performing model rather than simply the last training step.
- Test evaluation is kept separate from validation because the test set should remain the final unbiased estimate of generalization.


## Verification Plan

### Automated Checks

- Run the notebook end to end and verify dataset loading, model build, stage 1 training, stage 2 fine-tuning, and test evaluation complete without errors.
- Confirm validation accuracy surpasses the weak baseline and that the best checkpoint is written under `models/`.

### Manual Checks

- Inspect training and validation curves for overfitting signs.
- Visually inspect a few augmented samples to confirm sign semantics are preserved.
- Validate live inference separately with the saved `.keras` checkpoint.

## Expected Results

| Stage | Expected Validation Accuracy |
| --- | --- |
| Baseline | ~31% |
| Normalization fixed | ~50-60% |
| Better data + augmentation + callbacks | ~60-80% |
| Fine-tuning | ~75-90%+ |
